# Kerr–Newman black holes: horizons, ergosphere and naked singularities

*astro · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/kerr_newman_ergosphere)

The most general black hole of general relativity with electromagnetism, with mass $M$, spin $a$ and charge $Q$. For a whole batched grid of $(M, a, Q)$ the code computes the horizons $r_\pm$, the ergosphere, frame dragging and the equatorial ISCOs and photon orbits (metric derivatives by autograd). The animation sweeps spin and charge up to and past extremality $a^2 + Q^2 = M^2$, where the horizon disappears and the ring singularity would be exposed, which the cosmic censorship conjecture (unproven) says nature forbids.

**Vector form**

$$ds^2 = -\frac{\Delta}{\Sigma}\big(dt - a\sin^2\theta\,d\varphi\big)^2 + \frac{\sin^2\theta}{\Sigma}\big((r^2 + a^2)\,d\varphi - a\,dt\big)^2 + \frac{\Sigma}{\Delta}dr^2 + \Sigma\,d\theta^2,\qquad \Sigma = r^2 + a^2\cos^2\theta,\quad \Delta = r^2 - 2Mr + a^2 + Q^2$$

**Index (tensor) form**

$$r_{\pm,b} = M_b \pm \sqrt{M_b^2 - a_b^2 - Q_b^2}\qquad r_{E,b\theta} = M_b + \sqrt{M_b^2 - Q_b^2 - a_b^2\cos^2\theta}\qquad \omega_{br} = -\frac{g_{t\varphi}}{g_{\varphi\varphi}}\qquad \Omega_{\pm,br} = \frac{-\partial_r g_{t\varphi} \pm \sqrt{(\partial_r g_{t\varphi})^2 - \partial_r g_{tt}\,\partial_r g_{\varphi\varphi}}}{\partial_r g_{\varphi\varphi}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $M_b,\ a_b,\ Q_b$ | `tensors.parameter_grid()` | (3, n, n) | every combination of mass, spin and charge as one batch, $G = c = 1$ |
| $r_{\pm,b}$ | `M + torch.sqrt(M**2 - a**2 - Q**2)` | (3, n, n) | roots of $\Delta = 0$; NaN when $a^2 + Q^2 > M^2$ (naked) |
| $r_{E,b\theta}$ | `ergosphere(theta[None, :], M[:, None], a[:, None], Q[:, None])` | (F, T) | static limit $g_{tt} = 0$, broadcast frames $\times$ polar angles |
| $g_{tt},\ g_{t\varphi},\ g_{\varphi\varphi}$ | `metric_t_phi(r, theta, M, a, Q)` | (B, R) | the rotating $(t, \varphi)$ block of the Boyer–Lindquist metric |
| $\omega_{br}$ | `frame_dragging(r, theta, M, a, Q)` | (B, R) | angular velocity of a zero-angular-momentum observer; $\Omega_H = a/(r_+^2 + a^2)$ at the horizon |
| $\partial_r g_{\mu\nu}$ | `torch.autograd.grad(g.sum(), r)` | (B, R) | metric derivatives on the radius grid by autograd |
| $\Omega_{\pm,br},\ E_{\pm,br}$ | `circular_orbits(r, M, a, Q)` | (B, R) | circular-orbit angular velocity and energy; ISCO at the minimum of $E$, photon orbit where the orbit needs speed $c$ |
| $\kappa_b$ | `(r_plus - r_minus) / (2 * (r_plus**2 + a**2))` | (B,) | surface gravity, $T_H = \kappa/2\pi$; zero for an extremal hole |

Units $G = c = 1$: lengths in $GM/c^2$, spin $a = J/(Mc)$, charge in geometrised units. Established (classical GR): the Kerr–Newman metric (Newman et al. 1965) solves the Einstein–Maxwell equations, its horizons are the roots of $\Delta$, and the ergosphere, where no observer can stay at rest, lies between $r_+$ and $r_E(\theta)$, touching the horizon at the poles. Inside it frame dragging forces everything to co-rotate: the green curve falls in with zero angular momentum and is still swept around. Prograde orbits hug the hole: for Kerr the ISCO moves from $6M$ at $a = 0$ to $M$ (prograde) and $9M$ (retrograde) at $a = M$, matching Bardeen, Press and Teukolsky (1972), and the extremal Reissner–Nordström ISCO is $4M$. For $a^2 + Q^2 > M^2$ the formulas still give a metric, but with no horizon and a naked ring singularity at $r = 0$, $\theta = \pi/2$ (drawn in red). Penrose's weak cosmic censorship conjecture says gravitational collapse never produces one; it is unproven in general (known counterexamples need fine-tuned initial data or extra dimensions), and attempts to overspin or overcharge an extremal hole by throwing in particles appear to fail once second-order perturbation effects are included (Sorce and Wald 2017). Real holes have $Q \approx 0$ (charge is quickly neutralised), and measured spins (up to about $0.95$–$0.99\,M$) stay below extremality. The equatorial panel uses Boyer–Lindquist $r$ as the radius; there $\varphi$ of the infalling green curve winds up without bound at $r_+$, a coordinate effect.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/kerr_newman_ergosphere/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # float64 closed forms and a fine radius grid: the CPU is the right device here.
    M, a, Q = tensors.parameter_grid(n=params["grid"], masses=(1.0, 2.0, 5.0))  # (3, n, n): one batch
    g = solve.survey(M, a, Q)
    frac = g["naked"].double().mean().item()
    print(f"batched grid: {M.numel()} holes (M in {{1, 2, 5}}, a/M and Q/M in [0, 1.3]); "
          f"{100 * frac:.1f}% are naked (a² + Q² > M²)")  # fmt: skip
    scale = (g["r_plus"][1] / g["r_plus"][0]).nan_to_num(2.0)
    print(f"r₊ scales with M: r₊(2M)/r₊(M) in [{scale.min():.6f}, {scale.max():.6f}]")

    chi, Mf, af, Qf = tensors.sweep(params["angle"], params["chi_max"], params["frames"])
    theta = tensors.polar_angles()
    sw = solve.sweep(Mf, af, Qf, theta)

    keys = (
        "r_plus",
        "r_minus",
        "ergo_equator",
        "isco_pro",
        "isco_retro",
        "photon_pro",
        "photon_retro",
        "omega_H",
        "kappa",
    )
    print(" ".join(f"{h:>6}" for h in ("χ", "a", "Q", "r₊", "r₋", "r_E eq", "ISCO+", "ISCO−", "γ+", "γ−", "Ω_H", "κ")))
    for k in sorted({0, len(chi) // 4, len(chi) // 2, int((chi <= 1).sum()) - 1, len(chi) - 1}):
        row = [chi[k], af[k], Qf[k]] + [sw[key][k] for key in keys]
        print(" ".join(f"{float(v):6.3f}" for v in row) + ("   naked" if sw["naked"][k] else ""))
    one = Mf[:1]
    print(f"checks: Schwarzschild ISCO = {float(sw['isco_pro'][0]):.4f} M (exact 6), photon sphere "
          f"{float(sw['photon_pro'][0]):.4f} M (exact 3); extremal Kerr prograde ISCO (BPT) = "
          f"{float(equation.kerr_isco(one, one)):.3f} M")  # fmt: skip

    grid = {"r_plus": g["r_plus"][0].numpy(), "naked": g["naked"][0].numpy(), "top": a[0, -1, 0].item()}
    return plot.animate(chi, af, Qf, theta, sw, grid, params["angle"], out_dir / "kerr_newman.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/kerr_newman_ergosphere/equation.py
r"""Physics → tensor translation.

The Kerr–Newman metric (Newman et al. 1965) is the unique stationary black hole of Einstein–Maxwell theory with
mass M, angular momentum J = aM and charge Q. In Boyer–Lindquist coordinates (t, r, θ, φ), units G = c = 1
(and Gaussian units for Q), with

    Σ = r^2 + a^2\cos^2θ,        Δ = r^2 - 2Mr + a^2 + Q^2,

the (t, φ) block of the metric is

    g_{tt} = -\left(1 - \frac{2Mr - Q^2}{Σ}\right),\quad g_{tφ} = -\frac{a\sin^2θ\,(2Mr - Q^2)}{Σ},\quad
    g_{φφ} = \frac{(r^2 + a^2)^2 - a^2Δ\sin^2θ}{Σ}\sin^2θ

(and g_{rr} = Σ/Δ, g_{θθ} = Σ). Everything here is a broadcast over a batch of (M, a, Q) at once:

    horizons       Δ = 0:        r_± = M ± \sqrt{M^2 - a^2 - Q^2}
    ergosurface    g_{tt} = 0:   r_E(θ) = M + \sqrt{M^2 - Q^2 - a^2\cos^2θ}
    frame dragging ω = -g_{tφ}/g_{φφ} = \frac{a(2Mr - Q^2)}{(r^2 + a^2)^2 - a^2Δ\sin^2θ}
    naked          a^2 + Q^2 > M^2: Δ has no real root, no horizon hides the ring singularity Σ = 0

Equatorial circular orbits (θ = π/2) have angular velocity Ω = dφ/dt solving the geodesic condition
∂_r g_{tt} + 2Ω ∂_r g_{tφ} + Ω^2 ∂_r g_{φφ} = 0, i.e.

    Ω_± = \frac{-∂_r g_{tφ} ± \sqrt{(∂_r g_{tφ})^2 - ∂_r g_{tt}\,∂_r g_{φφ}}}{∂_r g_{φφ}}

(+ prograde, − retrograde; the derivatives come from autograd). The energy per unit mass is
E = -(g_{tt} + g_{tφ}Ω)/\sqrt{N} with N = -(g_{tt} + 2g_{tφ}Ω + g_{φφ}Ω^2): the photon orbit is where N → 0
(the orbit needs the speed of light) and the ISCO is where E(r) is minimal (dE/dr = 0).
"""

import math

import torch

TOL = 1e-12  # a² + Q² within TOL·M² of M² counts as extremal, not naked


def _disc(M, a, Q):
    d = M**2 - a**2 - Q**2
    return torch.where(d.abs() < TOL * M**2, torch.zeros_like(d), d)


def is_naked(M: torch.Tensor, a: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    """a² + Q² > M²: no horizon (cosmic censorship conjectures nature forbids this). Any shape → same shape."""
    return _disc(M, a, Q) < 0


def horizons(M, a, Q):
    """Outer and inner horizon r_± = M ± √(M² − a² − Q²); NaN when naked. Any (broadcast) shape."""
    root = torch.sqrt(_disc(M, a, Q))  # NaN for negative discriminant
    return M + root, M - root


def ergosphere(theta, M, a, Q, inner: bool = False):
    """Static limit g_tt = 0: r_E(θ) = M ± √(M² − Q² − a² cos²θ) (NaN where it does not exist)."""
    d = M**2 - Q**2 - (a * torch.cos(theta)) ** 2
    d = torch.where(d.abs() < TOL * M**2, torch.zeros_like(d), d)
    return M - torch.sqrt(d) if inner else M + torch.sqrt(d)


def metric_t_phi(r, theta, M, a, Q):
    """(g_tt, g_tφ, g_φφ) of Kerr–Newman in Boyer–Lindquist coordinates, broadcast over all inputs."""
    s2 = torch.sin(theta) ** 2
    sigma = r**2 + (a * torch.cos(theta)) ** 2
    delta = r**2 - 2 * M * r + a**2 + Q**2
    m = 2 * M * r - Q**2  # "mass function" 2Mr − Q²
    g_tt = -(1 - m / sigma)
    g_tp = -a * s2 * m / sigma
    g_pp = ((r**2 + a**2) ** 2 - a**2 * delta * s2) * s2 / sigma
    return g_tt, g_tp, g_pp


def frame_dragging(r, theta, M, a, Q):
    """ω = −g_tφ / g_φφ: the angular velocity dφ/dt of an observer with zero angular momentum (ZAMO)."""
    delta = r**2 - 2 * M * r + a**2 + Q**2
    return a * (2 * M * r - Q**2) / ((r**2 + a**2) ** 2 - a**2 * delta * torch.sin(theta) ** 2)


def surface_gravity(M, a, Q):
    """κ = (r₊ − r₋) / (2(r₊² + a²)): Hawking temperature T = κ/2π, zero for an extremal hole."""
    r_plus, r_minus = horizons(M, a, Q)
    return (r_plus - r_minus) / (2 * (r_plus**2 + a**2))


def kerr_isco(M, a, prograde: bool = True):
    """Bardeen–Press–Teukolsky (1972) closed form for Q = 0: r = M(3 + Z₂ ∓ √((3 − Z₁)(3 + Z₁ + 2Z₂)))."""
    x = a / M
    z1 = 1 + (1 - x**2) ** (1 / 3) * ((1 + x) ** (1 / 3) + (1 - x).clamp(min=0) ** (1 / 3))
    z2 = torch.sqrt(3 * x**2 + z1**2)
    root = torch.sqrt((3 - z1) * (3 + z1 + 2 * z2))
    return M * (3 + z2 - root) if prograde else M * (3 + z2 + root)


def circular_orbits(r, M, a, Q):
    """Equatorial circular geodesics on a radius grid. r: (B, R), M, a, Q: (B, 1).

    Returns dict of (B, R) tensors per branch ('pro', 'retro'): Omega = dφ/dt, N (> 0 ⇔ timelike), E.
    """
    r = r.detach().requires_grad_(True)
    half_pi = torch.full_like(r, math.pi / 2)
    g = metric_t_phi(r, half_pi, M, a, Q)
    # each g depends on its own r only, so the gradient of the sum is the elementwise derivative ∂_r g
    dg = [torch.autograd.grad(gi.sum(), r, retain_graph=True)[0] for gi in g]
    g_tt, g_tp, g_pp = (gi.detach() for gi in g)
    d_tt, d_tp, d_pp = dg
    root = torch.sqrt(d_tp**2 - d_tt * d_pp)
    out = {}
    for name, sign in (("pro", 1.0), ("retro", -1.0)):
        omega = (-d_tp + sign * root) / d_pp
        N = -(g_tt + 2 * g_tp * omega + g_pp * omega**2)
        E = -(g_tt + g_tp * omega) / torch.sqrt(N.clamp(min=1e-300))
        out[name] = {"Omega": omega, "N": N, "E": torch.where(N > 0, E, torch.full_like(E, math.inf))}
    return out


def equatorial_orbits(M, a, Q, n_r: int = 40001, r_max: float = 12.0):
    """Photon orbits and ISCOs (prograde and retrograde) for a batch of holes. M, a, Q: (B,) → dict of (B,).

    Grid: log-spaced r from just outside r₊ to r_max·M. NaN for naked singularities.
    """
    r_plus, _ = horizons(M, a, Q)
    r_in = torch.nan_to_num(r_plus, nan=1.0) * (1 + 1e-7)
    s = torch.linspace(0, 1, n_r, dtype=M.dtype, device=M.device)  # (R,)
    log_r = torch.log(r_in)[:, None] + s[None, :] * torch.log(r_max * M / r_in)[:, None]  # (B, R)
    r = torch.exp(log_r)
    orbits = circular_orbits(r, M[:, None], a[:, None], Q[:, None])
    naked = is_naked(M, a, Q)
    rows = torch.arange(len(M), device=M.device)
    result = {}
    for name, orb in orbits.items():
        # photon orbit: outermost grid point with N ≤ 0, then linear interpolation of N to zero
        inside = orb["N"] <= 0
        k = torch.where(inside.any(1), (inside * torch.arange(n_r, device=M.device)).amax(1), 0).clamp(max=n_r - 2)
        n0, n1 = orb["N"][rows, k], orb["N"][rows, k + 1]
        lr = log_r[rows, k] + (log_r[rows, k + 1] - log_r[rows, k]) * n0 / (n0 - n1)
        # no N ≤ 0 point: the photon orbit has merged with the horizon (extremal prograde orbits, at r = r₊)
        result[f"photon_{name}"] = torch.where(inside.any(1), torch.exp(lr), r_in)
        # ISCO: minimum of E(r), refined by a parabola through the three grid points around it
        E = orb["E"]
        j = E.argmin(1).clamp(1, n_r - 2)
        e_m, e_0, e_p = E[rows, j - 1], E[rows, j], E[rows, j + 1]
        shift = 0.5 * (e_m - e_p) / (e_m - 2 * e_0 + e_p)
        h = log_r[:, 1] - log_r[:, 0]
        result[f"isco_{name}"] = torch.exp(log_r[rows, j] + torch.nan_to_num(shift).clamp(-1, 1) * h)
    return {k: torch.where(naked, torch.full_like(v, math.nan), v) for k, v in result.items()}


def zero_angular_momentum_spiral(M, a, Q, r0: float = 10.0, n_r: int = 400):
    """A particle dropped from rest at infinity with L = 0 (E = 1) in the equatorial plane: φ(r) by quadrature.

    u^φ = −g_tφ/Δ and u^r = −√(g_φφ − Δ)/r, so dφ/dr = g_tφ r / (Δ √(g_φφ − Δ)): it is dragged around although
    it has no angular momentum (φ diverges logarithmically at r₊ in these coordinates). M, a, Q: (B,).
    Returns r (B, R) and φ (B, R), from r0 inward to 1.03 r₊.
    """
    r_plus, _ = horizons(M, a, Q)
    s = torch.linspace(0, 1, n_r, dtype=M.dtype, device=M.device)
    r = r0 * M[:, None] + (1.03 * r_plus - r0 * M)[:, None] * s[None, :]  # (B, R), decreasing
    _, g_tp, g_pp = metric_t_phi(r, torch.full_like(r, math.pi / 2), M[:, None], a[:, None], Q[:, None])
    delta = r**2 - 2 * M[:, None] * r + (a**2 + Q**2)[:, None]
    dphi_dr = g_tp * r / (delta * torch.sqrt(g_pp - delta))
    steps = 0.5 * (dphi_dr[:, 1:] + dphi_dr[:, :-1]) * r.diff(dim=1)  # trapezoid rule
    phi = torch.cat([torch.zeros_like(r[:, :1]), torch.cumsum(steps, 1)], 1)
    return r, phi


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/kerr_newman_ergosphere/tensors.py
"""Build the concrete tensors: a batched (M, a, Q) grid, the sweep path through extremality and angle grids.

Units G = c = 1: lengths in GM/c², spin a = J/(Mc), charge Q in geometrised units (Q² ↔ G Q²/(4πε₀ c⁴)).
"""

import math

import torch

F64 = torch.float64


def parameter_grid(n: int = 61, masses=(1.0, 2.0, 5.0), top: float = 1.3, device=None):
    """Every combination of mass, spin and charge as one batch: M, a, Q of shape (n_M, n, n).

    a/M and Q/M run from 0 to `top`, so the grid crosses the extremal circle a² + Q² = M².
    """
    m = torch.tensor(masses, dtype=F64, device=device)  # (n_M,)
    f = torch.linspace(0, top, n, dtype=F64, device=device)  # (n,) fractions of M
    M = m[:, None, None].expand(-1, n, n)  # (n_M, n, n)
    a = M * f[None, :, None]  # spin varies along axis 1
    Q = M * f[None, None, :]  # charge along axis 2
    return M, a, Q


def sweep(angle_deg: float, chi_max: float, n: int, device=None):
    """Path from Schwarzschild through extremality: χ = √(a² + Q²)/M from 0 to χ_max at a fixed mixing angle
    ψ (0° = pure spin, Kerr; 90° = pure charge, Reissner–Nordström). Returns χ, M, a, Q, each (n,)."""
    psi = math.radians(angle_deg)
    n_hole = max(2, round(n / chi_max))  # frames up to chi = 1, which is hit exactly (the extremal hole)
    chi = torch.cat([torch.linspace(0, 1, n_hole, dtype=F64, device=device),
                     torch.linspace(1, chi_max, n - n_hole + 1, dtype=F64, device=device)[1:]])  # fmt: skip
    M = torch.ones_like(chi)
    return chi, M, chi * math.cos(psi) * M, chi * math.sin(psi) * M


def polar_angles(n: int = 241, device=None) -> torch.Tensor:
    """θ from the north pole (0) to the south pole (π): (n,)."""
    return torch.linspace(0, math.pi, n, dtype=F64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/kerr_newman_ergosphere/solve.py
"""Evaluate every horizon, ergosurface, orbit and frame-dragging quantity for a whole batch of holes at once."""

import torch

from . import equation


def survey(M: torch.Tensor, a: torch.Tensor, Q: torch.Tensor) -> dict:
    """The batched grid (any shape): horizons, naked mask, surface gravity, equatorial ergosphere radius."""
    r_plus, r_minus = equation.horizons(M, a, Q)
    return {
        "r_plus": r_plus,
        "r_minus": r_minus,
        "naked": equation.is_naked(M, a, Q),
        "kappa": equation.surface_gravity(M, a, Q),
        "ergo_equator": equation.ergosphere(torch.full_like(M, torch.pi / 2), M, a, Q),
    }


def sweep(M: torch.Tensor, a: torch.Tensor, Q: torch.Tensor, theta: torch.Tensor) -> dict:
    """Everything the animation needs along the sweep. M, a, Q: (F,) frames; theta: (T,).

    Returns (F,) radii, (F, T) ergosurfaces, equatorial orbits, horizon spin Ω_H and the L = 0 spiral (F, R).
    """
    out = survey(M, a, Q)
    th, Mb, ab, Qb = theta[None, :], M[:, None], a[:, None], Q[:, None]  # broadcast (F, 1) × (1, T) → (F, T)
    out["ergo_outer"] = equation.ergosphere(th, Mb, ab, Qb)
    out["ergo_inner"] = equation.ergosphere(th, Mb, ab, Qb, inner=True)
    out.update(equation.equatorial_orbits(M, a, Q, n_r=8001))
    out["omega_H"] = a / (out["r_plus"] ** 2 + a**2)  # frame dragging at the horizon (rigid rotation)
    holes = ~out["naked"]
    out["spiral_r"], out["spiral_phi"] = equation.zero_angular_momentum_spiral(M, a, Q)
    out["spiral_phi"][~holes] = float("nan")
    # angular velocities dφ/dt of the circular orbits (for the moving particles), at their ISCO / photon radii
    for name in ("pro", "retro"):
        for kind in ("isco", "photon"):
            r = out[f"{kind}_{name}"]
            orb = equation.circular_orbits(torch.nan_to_num(r, nan=10.0)[:, None], Mb, ab, Qb)[name]
            out[f"Omega_{kind}_{name}"] = orb["Omega"][:, 0]
    return out


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/kerr_newman_ergosphere/plot.py
"""Animate the hole as spin and charge sweep past extremality: meridional cross-section, equatorial orbits,
the (a, Q) plane of the batched grid and the landmark radii."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

ERGO = "#fdba74"


def _np(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().numpy()


def _meridional(r: np.ndarray, theta: np.ndarray, a: float):
    """Oblate-spheroidal drawing coordinates x = √(r² + a²) sin θ, z = r cos θ (right half-plane)."""
    return np.sqrt(r**2 + a**2) * np.sin(theta), r * np.cos(theta)


def _closed(x: np.ndarray, z: np.ndarray):
    """Mirror the right half (θ: 0 → π) to the left so the outline closes."""
    return np.concatenate([x, -x[::-1]]), np.concatenate([z, z[::-1]])


def animate(chi, a, Q, theta, sw: dict, grid: dict, angle_deg: float, path: Path) -> Path:
    chi, a, Q, th = _np(chi), _np(a), _np(Q), _np(theta)
    s = {k: _np(v) for k, v in sw.items()}
    n = len(chi)
    frames = frame_indices(n, 100)

    # particle phases on the circular orbits: φ advances by Ω Δt per frame (coordinate time)
    dt = 2.2
    phase = {}
    for key in ("isco_pro", "isco_retro", "photon_pro", "photon_retro"):
        om = np.nan_to_num(s[f"Omega_{key}"])[frames]
        phase[key] = dict(zip(frames, np.cumsum(om) * dt))

    fig = plt.figure(figsize=(10, 5))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.05, 1])
    ax_m = fig.add_subplot(gs[:, 0])
    ax_e = fig.add_subplot(gs[:, 1])
    ax_q = fig.add_subplot(gs[0, 2])
    ax_r = fig.add_subplot(gs[1, 2])

    # --- (a, Q) plane: r₊/M of the M = 1 slice of the batched grid ---
    top = float(grid["top"])
    img = np.where(grid["naked"], np.nan, grid["r_plus"])
    cmap = plt.get_cmap("viridis", 10).copy()
    cmap.set_bad("#e5e7eb")
    im = ax_q.imshow(img.T, origin="lower", extent=[0, top, 0, top], cmap=cmap, vmin=1, vmax=2, aspect="equal")
    cb = fig.colorbar(im, ax=ax_q, fraction=0.046, pad=0.03)
    cb.set_label("r₊ / M", fontsize=7)
    cb.ax.tick_params(labelsize=6)
    ring = np.linspace(0, np.pi / 2, 100)
    ax_q.plot(np.cos(ring), np.sin(ring), color="black", lw=1)
    ax_q.text(0.92 * top, 0.92 * top, "naked", ha="right", va="top", fontsize=7, color="#6b7280")
    ax_q.plot(a, Q, color=ACCENT[1], lw=1)
    here = ax_q.plot([], [], "o", color=ACCENT[1], ms=5)[0]
    ax_q.set_xlabel("a / M", fontsize=7)
    ax_q.set_ylabel("Q / M", fontsize=7)
    ax_q.tick_params(labelsize=6)
    ax_q.set_title("batched grid: horizon radius", fontsize=8)

    # --- landmark radii along the sweep ---
    curves = [
        ("r_plus", "black", "-", "r₊"),
        ("r_minus", "#6b7280", "--", "r₋"),
        ("ergo_equator", "#ea580c", "-", "ergosphere (equator)"),
        ("isco_pro", ACCENT[0], "-", "ISCO pro"),
        ("isco_retro", ACCENT[3], "-", "ISCO retro"),
        ("photon_pro", ACCENT[0], ":", "photon pro"),
        ("photon_retro", ACCENT[3], ":", "photon retro"),
    ]
    for key, color, ls, label in curves:
        ax_r.plot(chi, s[key], color=color, ls=ls, lw=1.2, label=label)
    ax_r.axvline(1.0, color="#9ca3af", lw=0.8)
    ax_r.text(1.01, 8.6, "extremal", fontsize=6, color="#6b7280")
    now = ax_r.axvline(0, color=ACCENT[1], lw=1)
    ax_r.set_xlim(0, chi[-1])
    ax_r.set_ylim(0, 9.5)
    ax_r.set_xlabel("χ = √(a² + Q²) / M", fontsize=7)
    ax_r.set_ylabel("r / M", fontsize=7)
    ax_r.tick_params(labelsize=6)
    ax_r.legend(fontsize=5, loc="upper left", ncol=2, frameon=False)

    # --- meridional cross-section: persistent artists, updated per frame ---
    ax_m.set_xlim(-2.6, 2.6)
    ax_m.set_ylim(-2.6, 2.6)
    ax_m.set_aspect("equal")
    ax_m.set_xlabel("√(r² + a²) sin θ   (GM/c²)", fontsize=7)
    ax_m.set_ylabel("r cos θ", fontsize=7)
    ax_m.tick_params(labelsize=6)
    ax_m.text(0, -2.45, "orange: ergoregion,  red: ring singularity", ha="center", fontsize=6, color="#6b7280")
    empty = np.zeros((1, 2))
    ergo_m = [ax_m.add_patch(plt.Polygon(empty, color=ERGO)) for _ in range(2)]
    hole_m = ax_m.add_patch(plt.Polygon(empty, color="black"))
    inner_m = ax_m.plot([], [], color="white", ls="--", lw=0.8)[0]
    ring_m = ax_m.plot([], [], "o", color="#dc2626", ms=4, zorder=5)[0]

    # --- equatorial plane ---
    circle = np.linspace(0, 2 * np.pi, 200)
    ring_xy = lambda r: np.stack([r * np.cos(circle), r * np.sin(circle)], 1)  # noqa: E731
    ax_e.set_xlim(-10, 10)
    ax_e.set_ylim(-10, 10)
    ax_e.set_aspect("equal")
    ax_e.tick_params(labelsize=6)
    ax_e.set_xlabel("r cos φ  (Boyer–Lindquist r)", fontsize=7)
    ax_e.set_title("equatorial plane: ISCO (o) and photon orbits (*)\nblue prograde, purple retrograde", fontsize=8)
    ergo_e = ax_e.add_patch(plt.Polygon(empty, color=ERGO))
    hole_e = ax_e.add_patch(plt.Polygon(empty, color="black"))
    spiral = ax_e.plot([], [], color=ACCENT[2], lw=1.2, label="L = 0 infall (dragged)")[0]
    styles = {"isco_pro": (ACCENT[0], "--", "o"), "isco_retro": (ACCENT[3], "--", "o"),
              "photon_pro": (ACCENT[0], ":", "*"), "photon_retro": (ACCENT[3], ":", "*")}  # fmt: skip
    orbit_lines = {key: ax_e.plot([], [], color=c, ls=ls, lw=1)[0] for key, (c, ls, _) in styles.items()}
    particles = {key: ax_e.plot([], [], mk, color=c, ms=6)[0] for key, (c, _, mk) in styles.items()}
    naked_ring = ax_e.plot([], [], color="#dc2626", lw=1.5)[0]
    info = ax_e.text(0, -9.5, "", ha="center", fontsize=6)
    ax_e.legend(handles=[spiral], fontsize=6, loc="upper right", frameon=False)
    dk = 0.5 * (chi[1] - chi[0])

    def update(k: int):
        ak = float(a[k])
        naked = bool(s["naked"][k])
        outer, inner = s["ergo_outer"][k], s["ergo_inner"][k]
        ok = np.isfinite(outer)
        if naked:  # toroidal ergoregion between the inner and outer static limits, one ring per side
            if ok.any():
                xo, zo = _meridional(outer[ok], th[ok], ak)
                xi, zi = _meridional(inner[ok], th[ok], ak)
                for patch, sgn in zip(ergo_m, (1, -1)):
                    patch.set_xy(np.stack([sgn * np.concatenate([xo, xi[::-1]]), np.concatenate([zo, zi[::-1]])], 1))
            else:
                for patch in ergo_m:
                    patch.set_xy(empty)
            hole_m.set_xy(empty)
            inner_m.set_data([], [])
            ring_m.set_markersize(7)
            label = "naked singularity: no horizon"
        else:
            ergo_m[0].set_xy(np.stack(_closed(*_meridional(outer, th, ak)), 1))
            ergo_m[1].set_xy(empty)
            hole_m.set_xy(np.stack(_closed(*_meridional(np.full_like(th, s["r_plus"][k]), th, ak)), 1))
            inner_m.set_data(*_closed(*_meridional(np.full_like(th, s["r_minus"][k]), th, ak)))
            ring_m.set_markersize(3)
            label = "extremal black hole (r₊ = r₋)" if abs(chi[k] - 1) < dk else "black hole"
        ring_m.set_data([-ak, ak], [0, 0])
        ax_m.set_title(f"a = {ak:.2f} M,  Q = {Q[k]:.2f} M\n{label}", fontsize=8)

        rE = s["ergo_equator"][k]
        ergo_e.set_xy(ring_xy(rE) if np.isfinite(rE) else empty)
        if naked:
            hole_e.set_xy(empty)
            spiral.set_data([], [])
            for key in styles:
                orbit_lines[key].set_data([], [])
                particles[key].set_data([], [])
            naked_ring.set_data(*ring_xy(max(abs(ak), 0.05)).T)
            info.set_text("naked: weak cosmic censorship (a conjecture) forbids this")
        else:
            hole_e.set_xy(ring_xy(s["r_plus"][k]))
            r_s, phi_s = s["spiral_r"][k], s["spiral_phi"][k]
            spiral.set_data(r_s * np.cos(phi_s), r_s * np.sin(phi_s))
            for key in styles:
                r, p = s[key][k], phase[key][k]
                orbit_lines[key].set_data(*ring_xy(r).T)
                particles[key].set_data([r * np.cos(p)], [r * np.sin(p)])
            naked_ring.set_data([], [])
            info.set_text(
                f"horizon spin Ω_H = {s['omega_H'][k]:.3f},  surface gravity κ = {s['kappa'][k]:.3f}  (c³/GM)"
            )
        here.set_data([a[k]], [Q[k]])
        now.set_xdata([chi[k], chi[k]])

    fig.suptitle(f"Kerr–Newman sweep at mixing angle ψ = {angle_deg:g}° (0° Kerr, 90° Reissner–Nordström)", fontsize=9)
    fig.tight_layout()
    return save_gif(fig, update, frames + [frames[-1]] * 8, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'angle': 30.0,
    'chi_max': 1.15,
    'frames': 70,
    'grid': 61
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.kerr_newman_ergosphere")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.kerr_newman_ergosphere.main")
problem_dir = Path("problems/astro/kerr_newman_ergosphere")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
